# bronze_ingest

Respaldo manual de la **Tarea 1** del DAG. Parte A: carga manual de los Parquet que no pasan por el pipeline. Parte B: ingesta con DLT (la que ejecuta Airflow).

Servicios: Spark `spark://spark-master:7077`, MinIO `http://minio:9000`, catalogo Nessie `http://nessie:19120`.

# Bronze - Parte A: carga manual
Segun el enunciado, solo `users_2021` entra por el pipeline (DLT). El resto de Bronze se carga a mano:
`post_2020`, `post_2021` y `users_2020`.

Para no ahogar los equipos, cada Parquet original se recorta a un rango de meses (por defecto enero a
junio) con DuckDB, que filtra archivos grandes con memoria acotada, y luego se sube a MinIO (bucket `bronze`).
Rutas resultantes: `bronze/post/2020/post_2020.parquet`, `bronze/post/2021/post_2021.parquet`,
`bronze/users/2020/users_2020.parquet`.

In [ ]:
import os
import duckdb
import s3fs

RAW_DIR = os.getenv("RAW_DIR", "/data/raw")
OUT_DIR = os.getenv("PREPARED_DIR", "/data/prepared")
MES_DESDE = int(os.getenv("FECHA_DESDE_MES", "1"))
MES_HASTA = int(os.getenv("FECHA_HASTA_MES", "6"))
S3_ENDPOINT = os.getenv("S3_ENDPOINT", "http://minio:9000")
BUCKET = "bronze"

# (archivo origen, tabla en bronze, anio)
CARGAS_MANUALES = [
    ("posts_2020.parquet", "post", 2020),
    ("posts_2021.parquet", "post", 2021),
    ("users.parquet", "users", 2020),   # users.parquet trae todos los anios: se recorta a 2020
]

In [ ]:
def rango_fechas(anio, mes_desde, mes_hasta):
    """Devuelve [inicio, fin) del rango de meses, como texto UTC para SQL."""
    fin_anio, fin_mes = (anio + 1, 1) if mes_hasta == 12 else (anio, mes_hasta + 1)
    return f"{anio}-{mes_desde:02d}-01 00:00:00+00", f"{fin_anio}-{fin_mes:02d}-01 00:00:00+00"


def recortar_parquet(origen, destino, anio, mes_desde, mes_hasta):
    """Deja en `destino` solo las filas de `origen` creadas en [anio, meses]. Devuelve el numero de filas."""
    inicio, fin = rango_fechas(anio, mes_desde, mes_hasta)
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    con = duckdb.connect()
    con.execute("SET memory_limit='1GB'; SET threads=2; SET temp_directory='/tmp/duckdb_tmp'")
    con.execute(f"""
        COPY (
            SELECT * FROM read_parquet('{origen}')
            WHERE CreationDate >= TIMESTAMPTZ '{inicio}' AND CreationDate < TIMESTAMPTZ '{fin}'
        ) TO '{destino}' (FORMAT PARQUET, COMPRESSION SNAPPY)
    """)
    filas = con.execute(f"SELECT count(*) FROM read_parquet('{destino}')").fetchone()[0]
    con.close()
    return filas


def subir_a_minio(ruta_local, clave):
    """Sube el archivo al bucket bronze de MinIO (sobrescribe si ya existe: override)."""
    fs = s3fs.S3FileSystem(
        key=os.getenv("AWS_ACCESS_KEY_ID", "admin"),
        secret=os.getenv("AWS_SECRET_ACCESS_KEY", "password"),
        client_kwargs={"endpoint_url": S3_ENDPOINT, "region_name": "us-east-1"},
    )
    fs.put(ruta_local, f"{BUCKET}/{clave}")

In [ ]:
for archivo, tabla, anio in CARGAS_MANUALES:
    origen = os.path.join(RAW_DIR, archivo)
    destino = os.path.join(OUT_DIR, f"{tabla}_{anio}.parquet")
    clave = f"{tabla}/{anio}/{tabla}_{anio}.parquet"
    filas = recortar_parquet(origen, destino, anio, MES_DESDE, MES_HASTA)
    subir_a_minio(destino, clave)
    print(f"[bronze manual] s3://{BUCKET}/{clave}: {filas:,} filas (meses {MES_DESDE}-{MES_HASTA} de {anio})")

# Bronze - Parte B: ingesta con DLT (la que ejecuta Airflow)
Tarea 1 del DAG. DLT lee la fuente `users` (Parquet de StackOverflow), se queda con los registros
del anio indicado (por defecto 2021, de enero a junio) y los guarda en el bucket `bronze` de MinIO
en formato Parquet: `bronze/users/2021/users_2021.parquet`.

Escritura **override**: `write_disposition="replace"` y un nombre de archivo fijo, asi ejecutar el
pipeline varias veces sobrescribe el mismo archivo y no genera duplicados.

In [ ]:
import os
import datetime as dt
import dlt
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

# Los nombres de columna de Bronze se dejan tal cual vienen en la fuente (datos crudos)
os.environ.setdefault("SCHEMA__NAMING", "direct")

ORIGEN = os.getenv("USERS_SOURCE", "/data/raw/users.parquet")
ANIO = int(os.getenv("USERS_YEAR", "2021"))
MES_DESDE = int(os.getenv("FECHA_DESDE_MES", "1"))
MES_HASTA = int(os.getenv("FECHA_HASTA_MES", "6"))

In [ ]:
def rango_fechas(anio, mes_desde, mes_hasta):
    utc = dt.timezone.utc
    inicio = dt.datetime(anio, mes_desde, 1, tzinfo=utc)
    fin = dt.datetime(anio + 1, 1, 1, tzinfo=utc) if mes_hasta == 12 else dt.datetime(anio, mes_hasta + 1, 1, tzinfo=utc)
    return inicio, fin


@dlt.resource(name="users", write_disposition="replace")
def users():
    """Lee el Parquet de origen por lotes y entrega solo los usuarios creados en el rango."""
    inicio, fin = rango_fechas(ANIO, MES_DESDE, MES_HASTA)
    tipo = pa.timestamp("ms", tz="UTC")
    ini, fi = pa.scalar(inicio, type=tipo), pa.scalar(fin, type=tipo)
    pf = pq.ParquetFile(ORIGEN)
    for lote in pf.iter_batches(batch_size=250_000):
        t = pa.Table.from_batches([lote])
        sub = t.filter(pc.and_(pc.greater_equal(t["CreationDate"], ini), pc.less(t["CreationDate"], fi)))
        if sub.num_rows:
            yield sub

In [ ]:
# DLT guarda en <bucket>/<dataset_name>/<layout>. Con dataset_name="users" y este patron
# el archivo queda en bronze/users/2021/users_2021.parquet (siempre el mismo nombre: override).
destino = dlt.destinations.filesystem(
    layout=str(ANIO) + "/{table_name}_" + str(ANIO) + ".{ext}",
    warn_unsafe_layout_separators=False,
)
pipeline = dlt.pipeline(
    pipeline_name="bronze_users",
    destination=destino,
    dataset_name="users",
    pipelines_dir=os.getenv("DLT_PIPELINES_DIR", "/tmp/dlt_pipelines"),
)
info = pipeline.run(users(), loader_file_format="parquet")
print(info)
print(f"[bronze dlt] users {ANIO} (meses {MES_DESDE}-{MES_HASTA}) cargado en s3://bronze/users/{ANIO}/users_{ANIO}.parquet")